# Trade scheduling experiments
Coursework implementation: approximate and exact integer DP, synthetic validation, and an optional supplied-data TSLA evaluation. Run the notebook from the repository root. TSLA data is not bundled.


### Experiment cell 1


In [ ]:
import math, time
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

N, T, ALPHA, PI = 100_000, 10, 0.001, 0.5
DATA_PATH = Path('data/TSLA.csv')

def memory_after(previous_memory, order):
    return (int(previous_memory) + int(order) + 1) // 2

def fill_count(memory, order, alpha=ALPHA, pi=PI):
    return int(math.ceil(max(0.0, 1.0-alpha*(memory**pi))*order))

def simulate_schedule(schedule, alpha=ALPHA, pi=PI):
    memory=0; fills=[]; memories=[]
    for n in map(int,schedule):
        memory=memory_after(memory,n); memories.append(memory); fills.append(fill_count(memory,n,alpha,pi))
    return np.array(fills,dtype=int), np.array(memories,dtype=int)

def exact_fill_count(memory, order):
    # For N <= 100000, alpha*sqrt(memory) < 1.  Therefore
    # ceil((1 - sqrt(memory)/1000)*order)
    # = order - floor(order*sqrt(memory)/1000).
    # isqrt(order**2 * memory) gives floor(order*sqrt(memory)) exactly.
    order=int(order); memory=int(memory)
    loss=math.isqrt(order*order*memory)//1000
    return max(0, order-loss)

rng=np.random.default_rng(20260930)
test_memory=rng.integers(0,100001,size=100000)
test_order=rng.integers(0,100001,size=100000)
float_fill=np.ceil(np.maximum(0.0,1.0-0.001*np.sqrt(test_memory))*test_order).astype(np.int64)
integer_fill=np.maximum(0,test_order-np.floor(np.sqrt(test_order*test_order*test_memory)/1000.0).astype(np.int64))
assert np.array_equal(float_fill,integer_fill)
print('Exact integer fill identity: 100000 random checks passed')

print('Parameters:', N,T,ALPHA,PI)

### Experiment cell 2


In [ ]:
def approximate_dp(N,T,alpha=ALPHA,pi=PI,order_step=1000):
    units=N//order_step; states={(0,0):(0,())}; counts=[]
    for _ in range(T-1):
        nxt={}
        for (planned,memory),(value,path) in states.items():
            for du in range(units-planned+1):
                n=du*order_step; m2=memory_after(memory,n); v2=value+fill_count(m2,n,alpha,pi)
                key=(planned+du,m2); old=nxt.get(key)
                if old is None or v2>old[0]: nxt[key]=(v2,path+(n,))
        states=nxt; counts.append(len(states))
    best=(-1,None)
    for (planned,memory),(value,path) in states.items():
        n=N-planned*order_step; m2=memory_after(memory,n); total=value+fill_count(m2,n,alpha,pi)
        if total>best[0]: best=(total,path+(n,))
    fills,memories=simulate_schedule(best[1],alpha,pi)
    return {'schedule':best[1],'fills':fills,'memories':memories,'total':int(best[0]),'state_counts':counts}

approx=[]
for step in [5000,2500,1000]:
    t0=time.perf_counter(); r=approximate_dp(N,T,ALPHA,PI,step); elapsed=time.perf_counter()-t0
    approx.append((step,r,elapsed))
    print('step',step,'total',r['total'],'schedule',r['schedule'],'fills',r['fills'].tolist(),'states',r['state_counts'],'runtime',round(elapsed,3))

### Experiment cell 3


In [ ]:
def exact_dp(N,T,alpha=ALPHA,pi=PI, max_states=None):
    """Exact integer DP with dominance pruning over reachable states.

    State is (planned shares, current memory). For each planned amount x,
    a state (x,m1,v1) dominates (x,m2,v2) when m1 <= m2 and v1 >= v2.
    Since future memory is monotone in current memory and future fills are
    nonincreasing in memory, dominated states can never lead to a better
    completion. No order or memory grid is used; every integer action is
    considered before pruning.
    """
    states={(0,0):(0,())}; raw_counts=[]; kept_counts=[]
    integer_default_model=(abs(alpha-0.001)<1e-15 and abs(pi-0.5)<1e-15)
    reward=lambda memory,order: exact_fill_count(memory,order) if integer_default_model else fill_count(memory,order,alpha,pi)
    for period in range(T-1):
        raw={}
        for (planned,memory),(value,path) in states.items():
            for n in range(N-planned+1):
                m2=memory_after(memory,n); v2=value+reward(m2,n)
                key=(planned+n,m2); old=raw.get(key)
                if old is None or v2>old[0]: raw[key]=(v2,path+(n,))
        raw_counts.append(len(raw))
        by_planned={}
        for (planned,memory),(value,path) in raw.items():
            by_planned.setdefault(planned,[]).append((memory,value,path))
        states={}
        for planned,frontier in by_planned.items():
            best_value=-1
            for memory,value,path in sorted(frontier):
                if value>best_value:
                    states[(planned,memory)]=(value,path); best_value=value
        kept_counts.append(len(states))
        if max_states is not None and len(states)>max_states:
            raise MemoryError(f'period {period+1}: {len(states):,} nondominated states > cap {max_states:,}')
    best=(-1,None)
    for (planned,memory),(value,path) in states.items():
        n=N-planned; m2=memory_after(memory,n); total=value+reward(m2,n)
        if total>best[0]: best=(total,path+(n,))
    fills,memories=simulate_schedule(best[1],alpha,pi)
    return {'schedule':best[1],'fills':fills,'memories':memories,'total':int(best[0]),
            'raw_state_counts':raw_counts,'state_counts':kept_counts}

def brute_force(N,T,alpha=ALPHA,pi=PI):
    best=(-1,None)
    def rec(prefix,remaining,slots):
        nonlocal best
        if slots==1:
            s=tuple(prefix+[remaining])
            memory=0; val=0
            for n in s:
                memory=memory_after(memory,n); val+=exact_fill_count(memory,n)
            if val>best[0]: best=(val,s)
        else:
            for n in range(remaining+1): rec(prefix+[n],remaining-n,slots-1)
    rec([],N,T); return best

small=exact_dp(12,3); brute=brute_force(12,3); assert small['total']==brute[0]
print('Small exact validation:',small['total'],brute[0],small['schedule'])
medium=exact_dp(200,5,max_states=200000)
print('Medium exact diagnostic:',medium['total'],medium['schedule'])
print('Medium raw states:',medium['raw_state_counts'])
print('Medium nondominated states:',medium['state_counts'])

### Experiment cell 4


In [ ]:
def load_tsla(path):
    raw=pd.read_csv(path,low_memory=False); trade=raw.iloc[:,:8].copy(); trade.columns=trade.iloc[2].tolist(); trade=trade.iloc[3:].reset_index(drop=True)
    trade['Dates']=pd.to_datetime(trade['Dates'],format='%m/%d/%y %H:%M',errors='coerce'); trade['Volume']=pd.to_numeric(trade['Volume'],errors='coerce')
    return trade.dropna(subset=['Dates','Volume']).set_index('Dates').sort_index()

def make_blocks(trade):
    blocks=[]
    for day in sorted(trade.index.normalize().unique()):
        start=pd.Timestamp(day)+pd.Timedelta(hours=9,minutes=30); idx=pd.date_range(start,periods=120,freq='min'); v=trade['Volume'].reindex(idx)
        if v.notna().all(): blocks.append(v.to_numpy(float).reshape(12,10))
    return blocks

def avg_score(fills,blocks,days):
    return float(np.mean([np.minimum(fills,blocks[d][b]).sum() for d in range(days) for b in range(12)]))

trade=load_tsla(DATA_PATH); blocks=make_blocks(trade); days=len(blocks); half=days//2
RUN_EXACT_DEFAULT=False
if RUN_EXACT_DEFAULT:
    exact_default=exact_dp(N,T,ALPHA,PI,max_states=2000000)
    schedule_label='Exact DP, pi=0.5'
else:
    exact_default=approx[-1][1]
    schedule_label='Approximate DP step=1000'
    print('Using the finest approximate schedule for the default TSLA comparison; set RUN_EXACT_DEFAULT=True to attempt exact DP.')
exact_score=avg_score(exact_default['fills'],blocks,half)
equal=tuple([N//T]*(T-1)+[N-(N//T)*(T-1)]); equal_fills,_=simulate_schedule(equal,ALPHA,PI); equal_score=avg_score(equal_fills,blocks,half)
comparison=pd.DataFrame([{'schedule':schedule_label,'orders':exact_default['schedule'],'fills':exact_default['fills'].tolist(),'total_fills':exact_default['total'],'training_score':exact_score},{'schedule':'Equal-sized','orders':equal,'fills':equal_fills.tolist(),'total_fills':int(equal_fills.sum()),'training_score':equal_score}])
print('Complete days:',days,'training days:',half)
if RUN_EXACT_DEFAULT:
    print('Exact raw states:',exact_default['raw_state_counts']); print('Exact nondominated states:',exact_default['state_counts'])
else:
    print('Exact default-scale DP was not run; no exact state counts are reported.')
display(comparison)

### Experiment cell 5


In [ ]:
pi_rows=[]
for p in np.round(np.arange(.30,.701,.05),2):
    r=exact_dp(N,T,ALPHA,float(p),max_states=2000000) if RUN_EXACT_DEFAULT else approximate_dp(N,T,ALPHA,float(p),2500)
    train=avg_score(r['fills'],blocks,half); hold=avg_score(r['fills'],blocks[half:],days-half)
    pi_rows.append({'pi':float(p),'training_score':train,'holdout_score':hold,'total_fills':r['total'],'schedule':r['schedule']})
pi_table=pd.DataFrame(pi_rows); best=pi_table.loc[pi_table.training_score.idxmax()]
pi_table['method']='exact DP' if RUN_EXACT_DEFAULT else 'approximate DP, order step 2500'
display(pi_table[['pi','method','training_score','holdout_score','total_fills']])
print('Selected pi:',best['pi']); print('Selected schedule:',best['schedule']); print('Training score:',best['training_score']); print('Holdout score:',best['holdout_score'])
plt.figure(figsize=(7,4)); plt.plot(pi_table.pi,pi_table.training_score,'o-',label='training'); plt.plot(pi_table.pi,pi_table.holdout_score,'s-',label='holdout'); plt.xlabel('pi'); plt.ylabel('average interval score'); plt.title('Approximate-DP score versus pi' if not RUN_EXACT_DEFAULT else 'Exact-DP score versus pi'); plt.grid(alpha=.3); plt.legend(); plt.tight_layout(); plt.savefig('project1_score_vs_pi.png',dpi=160); plt.show()